# Visualizing catalysts' models

This notebook teaches you to build, optimize, and render atomistic models of electrocatalysts at a quality suitable for publication.

Three rendering approaches are covered:

1. Interactive 3D visualization via ASE's x3d viewer
2. Animated PNG sequences for comparing multiple structures side by side
3. Static high-resolution images rendered with POV-Ray.

The catalyst models covered are relevant to modern electrocatalysis research. Oxide surfaces are constructed by downloading experimental CIF files and substituting single metal atoms to simulate single-atom catalyst sites. Metal-nitrogen-carbon (M-N-C) materials are built programmatically from graphene nanoribbons by replacing specific carbon atoms with heteroatoms and inserting a transition metal center, then rendered as an animation cycling through 12 different metal and coordination combinations. A Pt(111) slab with an adsorbed OH-water adlayer models a platinum electrode under electrochemical conditions. A molecular FeN4 site and iron phthalocyanine (FePc) are compared as two realizations of the same Fe-N4 coordination motif in different structural contexts. Finally, a full oxygen reduction reaction (ORR) mechanism is modeled on Pt(111) by placing and optimizing all key intermediates of both the associative and dissociative pathways onto extended slabs and combining them into a single comprehensive visualization.Geometry optimizations throughout use TBLite with GFN1-xTB, and the BFGS or FIRE optimizers with FixAtoms constraints to freeze the substrate while relaxing only the adsorbates.

First, we need to install and load our software environment. We will use ASE to build catalyst models, TBLite as our semiempirical calculator, POV-Ray for publication-quality rendering, and the apng library to create animated PNG files. Please execute these cells to initialize the environment.

In [1]:
######################################################
##-install the necessery packages for this tutorial-##
######################################################

!apt install povray
!pip install ase tblite apng

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  fonts-povray libboost-thread1.74.0 libilmbase25 libopenexr25 libsdl1.2debian
  povray-includes
Suggested packages:
  povray-doc povray-examples
The following NEW packages will be installed:
  fonts-povray libboost-thread1.74.0 libilmbase25 libopenexr25 libsdl1.2debian
  povray povray-includes
0 upgraded, 7 newly installed, 0 to remove and 3 not upgraded.
Need to get 3,010 kB of archives.
After this operation, 11.5 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 fonts-povray all 1:3.7.0.10-1 [70.0 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 libboost-thread1.74.0 amd64 1.74.0-14ubuntu3 [262 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libilmbase25 amd64 2.5.7-2 [175 kB]
Get:4 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libopenexr25 amd64 2.5

In [2]:
#########################
##-Importing libraries-##
#########################

##-System
import os

##-Data
import numpy as np

##-Visualization
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from apng import APNG, PNG

##-ASE - structure building
from ase import Atom, Atoms
from ase.build import graphene_nanoribbon, fcc111

##-ASE - I/O and visualization
from ase.io import read, write
from ase.visualize import view
from ase.io.pov import get_bondpairs

##-ASE - simulation
from ase.optimize import FIRE, BFGS
from ase.constraints import FixAtoms

##-Calculator
from tblite.ase import TBLite

Povray settings

In [3]:
povray_settings = {
    'display': False, ##-display while rendering
    'transparent': False, ##-transparency option
    'camera_type': 'perspective', ##-try 'orthographic', 'perspective' or 'ultra_wide_angle'
    'canvas_width': 4096, ##-width  of canvas in pixels
    'canvas_height': None, ##-height of canvas in pixels
    'image_plane': None, ##-Distance from front atom to image plane
    'camera_dist': 1000, ##-camera distance
	  'depth_cueing': False, ##-fade lines and shading as it gets further from the projection plane of the view
    'point_lights': [], ##-try [(0,0,100), 'White'],[(40,40,100),'White'], # [[loc1, color1], [loc2, color2],...]
    'area_light': [(0, 0, 100), 'White', 1000, 1000, 1, 1], ##-Location, color, [width, height, Nlamps_x, Nlamps_y]
    'celllinewidth': 0.05, ##-radius of the cylinders representing the cell
}

# Oxide catalysts

Oxides are a class of materials with remarkable catalytic activity for the oxygen reduction reaction (ORR). Here we load two experimentally relevant structures from a crystal structure database and manually substitute one atom with Platinum to model a single-atom Pt site embedded in the oxide surface. This mimics catalysts reported in recent literature where isolated Pt atoms paired with surface metal atoms show ultralow Pt loading with high activity.

In [4]:
%%capture
##-Gao, Ruijie, et al. "Pt/Fe2O3 with Pt–Fe pair sites as a catalyst for oxygen reduction with ultralow Pt loading." Nature Energy 6.6 (2021): 614-623.
!wget https://raw.githubusercontent.com/vilab-tartu/test_models/refs/heads/main/Oxides/Fe2O3.cif
Fe2O3 = read('Fe2O3.cif')
Fe2O3 = Fe2O3*(3,3,1)
Fe2O3[32].symbol = 'Pt'
write('Fe2O3.xyz',Fe2O3*(3,3,1))

In [5]:
view(Fe2O3,viewer='x3d')

In [6]:
%%capture
##-Liu, Fan, et al. Avoiding Sabatier’s Limitation on Spatially Correlated Pt–Mn Atomic Pair Sites for Oxygen Electroreduction. Journal of the American Chemical Society 145(46) (2023): 25252-25263.
!wget https://raw.githubusercontent.com/vilab-tartu/test_models/refs/heads/main/Oxides/MnO2.cif
MnO2 = read('MnO2.cif')
MnO2 = MnO2*(2,9,1)
MnO2[77].symbol = 'Pt'
write('MnO2.xyz',MnO2*(3,12,3))

In [7]:
view(MnO2, viewer='x3d')

# Model Metal-Nitrogen-Carbon (M-N-C) catalysts

M-N-C materials are a promising class of non-precious metal catalysts for the ORR. The active site consists of a single transition metal atom coordinated by nitrogen atoms in a graphene-like carbon matrix. The mnc_model() function builds this structure from a graphene nanoribbon by replacing specific carbon atoms with heteroatoms and inserting the metal at the center. Here we generate a series of 12 different M-N-C configurations, varying the metal (Mn, Fe, Ni, Co) and the coordinating atoms (N, C, S), render each one with POV-Ray, and compile them into an animated PNG to compare geometries side by side. You can convert the APNG to GIF at https://ezgif.com/apng-to-gif.

In [8]:
def mnc_model(A0,A1,A2,A3,A4):
  MNC = graphene_nanoribbon(4, 2, type='armchair', saturated=False, vacuum=3.5)

  MNC.append(Atom(A0, position=(MNC[11].position+MNC[12].position)/2))
  MNC[10].symbol = A1
  MNC[13].symbol = A2
  MNC[18].symbol = A3
  MNC[21].symbol = A4

  del MNC[[11,12,24,27,28,31]]
  write(f'{A0}{A1}{A2}{A3}{A4}.xyz',MNC)

  generic_projection_settings = {
      'rotation': '90x,0y,0z', ##-try'30x,30y,0z'
      'radii': 0.9, ##-float, or a list with one float per atom
      'colors': None, ##-List: one (r, g, b, t) tuple per atom
      'show_unit_cell': 0, ##-0, 1, or 2 to not show, show, and show all of cell
  }

  textures = ['jmol' for atom in MNC] ##-try options: jmol, ase3, vmd, glass, pale, simple

  write(f'{A0}{A1}{A2}{A3}{A4}.pov',MNC, **generic_projection_settings, povray_settings=povray_settings)

In [9]:
systems = [
    ['Mn','N','N','N','N'],
    ['Fe','N','N','N','N'],
    ['Ni','N','N','N','N'],
    ['Co','N','N','N','N'],
    ['Mn','C','C','N','N'],
    ['Fe','C','N','C','N'],
    ['Ni','C','N','N','C'],
    ['Co','C','C','C','C'],
    ['Mn','S','N','N','N'],
    ['Fe','N','S','N','N'],
    ['Ni','N','N','S','N'],
    ['Co','N','N','N','S'],
]

for elements in systems:
  A0 = elements[0]
  A1 = elements[1]
  A2 = elements[2]
  A3 = elements[3]
  A4 = elements[4]

  mnc_model(A0,A1,A2,A3,A4)

  name = f'{A0}{A1}{A2}{A3}{A4}'

  os.system(f'povray +I{name}.pov +O{name}.png +A +AM2 +UA +Q9 -D')

In [10]:
##-Define the scaling factor
scaling_factor = 7 / 10

##-Generate list of expected PNG filenames
png_files = [f"{''.join(elements)}.png" for elements in systems]

##-Resize the width of each PNG file and save them
for file in png_files:
  if os.path.exists(file):
    with Image.open(file) as img:
      ##-Calculate the new width
      new_width = int(img.width * scaling_factor)
      new_height = img.height

      ##-Resize the image
      resized_img = img.resize((new_width, new_height), Image.LANCZOS)

      ##-Save the resized image, overwriting the original file
      resized_img.save(file)

print("All specified PNG files have been resized.")

##-Filter out files that do not exist
png_files = [file for file in png_files if os.path.exists(file)]
png_files.sort() ##-Sort files to maintain order

All specified PNG files have been resized.


In [11]:
##-Create an APNG object
apng = APNG()

##-Add each PNG file to the APNG with a 0.5 second delay
for file in png_files:
  png = PNG.open(file)
  apng.append(png, delay=500) ##-delay in milliseconds

##-Save the APNG
apng.save("model_MNC_animated.png")

print("Animated PNG saved as animated.png")

Animated PNG saved as animated.png


You can use https://ezgif.com/apng-to-gif to convert to animated gjf.

## OH–H₂O adlayer on Pt(111)

Here we load a pre-optimized Pt(111) slab with an adsorbed OH group and water molecules, a realistic model of a platinum electrode surface under electrochemical conditions. We visualize it interactively and then render a high-quality static image with POV-Ray. Try changing the rotation parameter in generic_projection_settings to view the slab from different angles, or the textures option to change the visual style.

In [12]:
!wget https://github.com/double-layer/test_models/raw/refs/heads/main/Pt-w/Pt-OH-H2O.traj
Pt111 = read('Pt-OH-H2O.traj',-1)
view(Pt111, viewer='x3d')

--2026-06-22 06:05:59--  https://github.com/double-layer/test_models/raw/refs/heads/main/Pt-w/Pt-OH-H2O.traj
Resolving github.com (github.com)... 140.82.114.4
Connecting to github.com (github.com)|140.82.114.4|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://github.com/doublelayer/test_models/raw/refs/heads/main/Pt-w/Pt-OH-H2O.traj [following]
--2026-06-22 06:05:59--  https://github.com/doublelayer/test_models/raw/refs/heads/main/Pt-w/Pt-OH-H2O.traj
Reusing existing connection to github.com:443.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/doublelayer/test_models/refs/heads/main/Pt-w/Pt-OH-H2O.traj [following]
--2026-06-22 06:05:59--  https://raw.githubusercontent.com/doublelayer/test_models/refs/heads/main/Pt-w/Pt-OH-H2O.traj
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (r

In [13]:
generic_projection_settings = {
    'rotation': '-80x,10y,0z', ##-try'30x,30y,0z'
    'radii': 1.1, ##-float, or a list with one float per atom
    'colors': None, ##-List: one (r, g, b, t) tuple per atom
    'show_unit_cell': 2, ##-0, 1, or 2 to not show, show, and show all of cell
}

povray_settings['textures'] = ['intermediate' for atom in Pt111] ##-try options: jmol, ase2, ase3, vmd, glass, pale, simple
write(f'Pt111.pov', Pt111, **generic_projection_settings, povray_settings=povray_settings)
os.system(f'povray +IPt111.pov +Omodel_Pt111.png +A +AM2 +UA +Q11 +W1024 +H3072')

0

## OH on a metal-Nitrogen-Carbon (M-N-C) catalyst

Using the same mnc_model() function, we now build an FeN4 active site and optimize an OH adsorbate above the Fe center using BFGS with GFN1-xTB. The carbon framework atoms are frozen with FixAtoms so only the adsorbate relaxes. The optimized structure is then rendered with POV-Ray.

In [14]:
def mnc_model(A0,A1,A2,A3,A4):
  MNC = graphene_nanoribbon(4, 2, type='armchair', saturated=False, vacuum=3.5)

  MNC.append(Atom(A0, position=(MNC[11].position+MNC[12].position)/2))
  MNC[10].symbol = A1
  MNC[13].symbol = A2
  MNC[18].symbol = A3
  MNC[21].symbol = A4

  del MNC[[11,12,24,27,28,31]]
  return MNC

In [15]:
FeN4 = mnc_model('Fe','N','N','N','N')

FeN4.center(vacuum=6.0, axis=1)

Fe_index = [i for i, a in enumerate(FeN4) if a.symbol == 'Fe'][0]
FeN4.set_constraint(FixAtoms(indices=list(range(len(FeN4)))))

r_Fe = FeN4[Fe_index].position
r_O = r_Fe + np.array([0.0, 1.8, 0.0])
r_H = r_O  + np.array([0.5, 0.5, 0.0])

FeN4 += Atoms('OH', positions=[r_O, r_H])

FeN4.calc = TBLite(method='GFN1-xTB')

BFGS(FeN4, trajectory='FeN4.traj', logfile='FeN4.log').run(fmax=0.05)

------------------------------------------------------------
  cycle        total energy    energy error   density error
------------------------------------------------------------
      1     -57.52740592669  -5.8430810E+01   9.9816319E-01
      2     -36.24197017713   2.1285436E+01   1.2147994E+00
      3     -39.14060450599  -2.8986343E+00   9.1886952E-01
      4     -56.69244970678  -1.7551845E+01   7.3726578E-01
      5     -58.47773955234  -1.7852898E+00   6.5466407E-01
      6     -57.96892840464   5.0881115E-01   6.2753886E-01
      7     -65.55966452798  -7.5907361E+00   3.1582833E-01
      8     -44.45452736386   2.1105137E+01   9.1451704E-01
      9     -64.88649094283  -2.0431964E+01   3.1718745E-01
     10     -63.64776318275   1.2387278E+00   3.9392048E-01
     11     -66.52924268037  -2.8814795E+00   1.2181484E-01
     12     -66.61833806335  -8.9095383E-02   9.9175766E-02
     13     -66.67358350660  -5.5245443E-02   8.5088110E-02
     14     -66.77988601199  -1.063025

np.True_

In [16]:
view(FeN4, viewer='x3d')

In [17]:
generic_projection_settings = {
    'rotation': '15x,10y,0z', ##-try'30x,30y,0z'
    'radii': 1.1, ##-float, or a list with one float per atom
    'colors': None, ##-List: one (r, g, b, t) tuple per atom
    'show_unit_cell': 2, ##-0, 1, or 2 to not show, show, and show all of cell
}

povray_settings['textures'] = ['intermediate' for atom in FeN4] ##-try options: jmol, ase2, ase3, vmd, glass, pale, simple

write(f'FeN4.pov', FeN4, **generic_projection_settings, povray_settings=povray_settings)

os.system(f'povray +IFeN4.pov +Omodel_FeN4.png +A +AM2 +UA +Q11 +W1024 +H3072')

0

## OH on iron phthalocyanine

Iron phthalocyanine (FePc) is a molecular catalyst with a planar Fe–N4 coordination environment, structurally similar to M-N-C but as a discrete molecule rather than a periodic framework. We download the structure from a repository, add an OH adsorbate above the Fe center, and optimize it the same way as FeN4. Compare the two active sites: both have Fe coordinated by four nitrogens, but the ligand environment and geometry differ

In [18]:
!wget https://gitlab.com/doublelayer/test-models/-/raw/main/iron-phthalocyanine-FePc-OH/FePc.xyz

FePc = read('FePc.xyz')

Fe_index = [i for i, a in enumerate(FePc) if a.symbol == 'Fe'][0]
FePc.set_constraint(FixAtoms(indices=list(range(len(FePc)))))

r_Fe = FePc[Fe_index].position
r_O = r_Fe + np.array([0.0, 0.0, 1.8])
r_H = r_O  + np.array([0.5, 0.0, 0.5])

FePc += Atoms('OH', positions=[r_O, r_H])

FePc.calc = TBLite(method='GFN1-xTB')

BFGS(FePc, trajectory='FePc.traj', logfile='FePc.log').run(fmax=0.05)

--2026-06-22 06:06:23--  https://gitlab.com/doublelayer/test-models/-/raw/main/iron-phthalocyanine-FePc-OH/FePc.xyz
Resolving gitlab.com (gitlab.com)... 172.65.251.78, 2606:4700:90:0:f22e:fbec:5bed:a9b9
Connecting to gitlab.com (gitlab.com)|172.65.251.78|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3210 (3.1K) [text/plain]
Saving to: ‘FePc.xyz’

FePc.xyz            100%[===================>]   3.13K  --.-KB/s    in 0s      

2026-06-22 06:06:24 (53.9 MB/s) - ‘FePc.xyz’ saved [3210/3210]

------------------------------------------------------------
  cycle        total energy    energy error   density error
------------------------------------------------------------
      1     -107.8164078044  -1.0950980E+02   8.0146863E-01
      2     -98.79107310257   9.0253347E+00   6.3039088E-01
      3     -59.90652459739   3.8884549E+01   8.4692723E-01
      4     -79.98693421335  -2.0080410E+01   6.5386801E-01
      5     -109.5647398497  -2.9577806E+01   1.9744956E

np.True_

In [19]:
view(FePc, viewer='x3d')

In [20]:
generic_projection_settings = {
    'rotation': '-80x,10y,0z', ##-try'30x,30y,0z'
    'radii': 1.1, ##-float, or a list with one float per atom
    'colors': None, ##-List: one (r, g, b, t) tuple per atom
    'show_unit_cell': 0, ##-0, 1, or 2 to not show, show, and show all of cell
}

povray_settings['textures'] = ['intermediate' for atom in FePc] ##-try options: jmol, ase2, ase3, vmd, glass, pale, simple

write(f'FePc.pov', FePc, **generic_projection_settings, povray_settings=povray_settings)

os.system(f'povray +IFePc.pov +Omodel_FePc.png +A +AM2 +UA +Q11 +W2048 +H1024')

0

# Catalysis mechanisms

This part takes time. Over 30 min.

## Create slab

To model the ORR mechanism on Pt(111), we first build a large periodic slab using fcc111. The slab geometry is controlled by a (number of unit cells along x), b (along y), c (number of layers), and v (vacuum thickness). All Pt atoms are frozen with FixAtoms, only the adsorbates will be allowed to move during optimization. We also compute the nearest-neighbor metal–metal distance M_M, which we use as a geometric reference to place adsorbates at physically meaningful positions.

In [21]:
##-variables to define the geometry of the model
a = 16
b = 4
c = 1
v = 8

In [22]:
slab = fcc111('Pt', size=(a, b, c), vacuum=v, orthogonal=True)

In [23]:
##-Adjust and constrain the geometry of the slab
constraint = FixAtoms(indices=[atom.index for atom in slab if atom.symbol == 'Pt'])
slab.set_constraint(constraint)

slab.center()

slab.positions[:, 2] += -v/2

In [24]:
##-Define metal-metal distance in Ångströms
distances = slab.get_all_distances()
M_M = np.min(distances[np.nonzero(distances)])

## Model associative pathway

In the associative ORR mechanism, O₂ is reduced without fully breaking the O–O bond first. The key intermediates are O₂, OOH, O, and OH, each adsorbed at a specific surface site. We place all four intermediates simultaneously on the slab and optimize with the FIRE algorithm using a loose convergence threshold (fmax=5), enough to relax the geometry without spending excessive time. This gives us a single snapshot model of the associative pathway for visualization purposes.

In [25]:
assoc = slab.copy()

##-Define the adsorbates and manually shift their coordinates
##-O2 at bridge site
o2 = Atoms('O2', positions=[(0, 0, 0), (-0.7, -1, 0)])
o2.positions += [2*M_M, np.sqrt(3)*M_M, 1.5] + assoc[a*b*(c-1)].position
assoc += o2

##-OOH at ontop site
ooh = Atoms('OOH', positions=[(0, 0, 0), (0, 0, 1.4), (0.77, 0, 2.1)])
ooh.positions += [5*M_M, np.sqrt(3)*M_M, 1.7] + assoc[a*b*(c-1)].position
assoc += ooh

##-O at hcp site
o = Atoms('O', positions=[(0, 0, 0)])
o.positions += [7.5*M_M, np.sqrt(3)*M_M-0.3*M_M, 1.6] + assoc[a*b*(c-1)+1].position
assoc += o

##-OH at ontop site
oh = Atoms('OH', positions=[(0, 0, 0), (0.5, -0.4, 0.96)])
oh.positions += [11*M_M, np.sqrt(3)*M_M, 1.7] + assoc[a*b*(c-1)+1].position
assoc += oh

In [26]:
%%time

##-this calculation takes <5 mins
assoc.calc = TBLite(method='GFN1-xTB',accuracy=1,max_iterations=500)
opt = FIRE(assoc, trajectory='assoc.traj', logfile='assoc.log')
opt.run(fmax=5)

------------------------------------------------------------
  cycle        total energy    energy error   density error
------------------------------------------------------------
      1     -307.4717563124  -3.0733613E+02   9.7099799E-01
      2      1464.290218280   1.7717620E+03   2.2118194E+00
      3      2397.295519613   9.3300530E+02   2.5497114E+00
      4      1829.601531611  -5.6769399E+02   2.4457484E+00
      5      2967.074967083   1.1374734E+03   2.6669892E+00
      6      2030.524104524  -9.3655086E+02   2.3394155E+00
      7      182.8871129602  -1.8476370E+03   1.7297270E+00
      8      842.1914962652   6.5930438E+02   2.0635184E+00
      9      933.8277133744   9.1636217E+01   1.9058925E+00
     10      936.3474719097   2.5197585E+00   1.8654375E+00
     11      114.8948767793  -8.2145260E+02   1.7089015E+00
     12      1087.005639214   9.7211076E+02   1.9259349E+00
     13      483.9547687130  -6.0305087E+02   1.4951702E+00
     14      730.3846355867   2.464298

np.True_

In [27]:
assoc=read('assoc.traj',-1)
view(assoc, viewer='x3d')

## Model dissociative pathway

In the dissociative pathway, O₂ splits immediately upon adsorption into two separate O atoms. The intermediates are then O+O, OOH, OH+OH, and H₂O. We build and optimize this pathway the same way as the associative one. Comparing the two pathways side by side is the goal of the next section.

In [28]:
disso = slab.copy()

In [29]:
##-Define the adsorbates and manually shift their coordinates
##-O + O at hollow sites
o2_ad = Atoms('O2', positions=[(0, 0, 0), (M_M, 0, 0)])
o2_ad.positions += [2.5*M_M, np.sqrt(3)*M_M - 0.3*M_M, 1.5] + disso[a*b*(c-1)].position
disso += o2_ad

##-OOH at ontop site
ooh = Atoms('OOH', positions=[(0, 0, 0), (M_M, 0, 0), (0.77, 0, 0.8)])
ooh.positions += [6*M_M, np.sqrt(3)*M_M, 1.7] + disso[a*b*(c-1)].position
disso += ooh

##-OH + OH at hcp site
o2h2 = Atoms('O2H2', positions=[(0, 0, 0), (M_M, 0, 0), (0.5, -0.4, 0.96), (M_M + 0.5, -0.4, 0.96)])
o2h2.positions += [9*M_M, np.sqrt(3)*M_M, 1.7] + disso[a*b*(c-1)+1].position
disso += o2h2

##-H2O at ontop site
h2o = Atoms('OH2', positions=[(0, 0, 0), (0.5, -0.2, 0.7), (-0.5, -0.2, 0.7)])
h2o.positions += [13*M_M, np.sqrt(3)*M_M, 1.9] + assoc[a*b*(c-1)+1].position
disso += h2o

In [ ]:
%%time
##-this calculations takes 5 mins
disso.calc = TBLite(method='GFN1-xTB',accuracy=1,max_iterations=500)
opt = FIRE(disso, trajectory='disso.traj', logfile='disso.log')
opt.run(fmax=5)

------------------------------------------------------------
  cycle        total energy    energy error   density error
------------------------------------------------------------
      1     -314.6313963644  -3.1451073E+02   9.6005441E-01
      2      1970.682507715   2.2853139E+03   2.2795873E+00
      3      2354.335717157   3.8365321E+02   2.5144550E+00
      4      729.0219827960  -1.6253137E+03   2.3718628E+00
      5      349.9223803077  -3.7909960E+02   1.7158116E+00
      6      820.0657109376   4.7014333E+02   1.8486016E+00
      7      1931.094884009   1.1110292E+03   2.2731677E+00
      8      1542.517429200  -3.8857745E+02   2.1217250E+00
      9      1802.269956576   2.5975253E+02   2.2584121E+00
     10      904.9969367069  -8.9727302E+02   1.8756008E+00
     11      801.2424544232  -1.0375448E+02   1.8574627E+00
     12      488.7123661495  -3.1253009E+02   1.6089239E+00
     13     -117.0009535913  -6.0571332E+02   1.2813374E+00
     14      590.6650752885   7.076660

In [ ]:
disso=read('disso.traj',-1)
view(disso, viewer='x3d')

## Model the combined path

Here we merge both pathways onto a single extended slab by concatenating the associative and dissociative models side by side, then adding gas-phase molecules (O₂ and H₂O) above the surface to complete the picture. This creates a single comprehensive visualization of the full ORR mechanism on Pt(111).

In [ ]:
model = assoc.copy()
back  = disso.copy()

back.positions[:, 1]  += 2*np.sqrt(3)*M_M
model += back

##-Adjust the unit cell
cell = model.get_cell()
cell[1, 1] *= 2
model.set_cell(cell)

model.center()
model.rotate(180,'z')
model.wrap()

##-H2O to the left of the slab
h2o = Atoms('OH2', positions=[(0, 0, 0), (-0.4, 0.4, 0.7), (0.4, -0.4, 0.7)])
h2o.positions += [-2*M_M, 5*np.sqrt(3)/2*M_M, 5] + assoc[a*b*(c-1)+1].position
model += h2o

##-O2 to the right of the slab
o2 = Atoms('O2', positions=[(0, 0, 0), (0, 0, 1.23)])
o2.positions += [16.5*M_M, 1*M_M, 5] + assoc[a*b*(c-1)].position
model += o2

##-H2O to the top of the slab
h2o = Atoms('OH2', positions=[(0, 0, 0), (-0.4, -0.4, 0.7), (0.4, 0.4, 0.7)])
h2o.positions += [5*M_M, 8*np.sqrt(3)/2*M_M, 5] + assoc[a*b*(c-1)].position
model += h2o

##-H2O to the top of the slab
h2o = Atoms('OH2', positions=[(0, 0, 0), (-0.4, -0.4, 0.7), (0.4, 0.4, 0.7)])
h2o.positions += [8.5*M_M, 8*np.sqrt(3)/2*M_M, 5] + assoc[a*b*(c-1)].position
model += h2o

##-O2 to the right of the slab
h2o2 = Atoms('O2H2', positions=[(0, 0, 0), (1.23, 0, 0), (-0.9, 0, 0), (1.23, 0, 0.9)])
h2o2.positions += [12*M_M, 8*np.sqrt(3)/2*M_M, 5] + assoc[a*b*(c-1)].position
model += h2o2

model.pbc = [False, False, False]

##-View the final concatenated model
view(model, viewer='x3d')

In [ ]:
%%time
##-this takes <30 min
constraint = FixAtoms(indices=[atom.index for atom in model if atom.symbol == 'Pt'])
model.set_constraint(constraint)

model.calc = TBLite(method='GFN1-xTB',accuracy=1,max_iterations=500)

opt = FIRE(model, trajectory='model.traj', logfile='model.log')
opt.run(fmax=15)

------------------------------------------------------------
  cycle        total energy    energy error   density error
------------------------------------------------------------
      1     -662.5055224538  -6.6258315E+02   9.3379257E-01
      2      869.8245617774   1.5323301E+03   1.8873608E+00
      3      2329.227799507   1.4594032E+03   2.2791719E+00
      4      2225.230325169  -1.0399747E+02   2.2395017E+00
      5      3780.893349109   1.5556630E+03   2.4544626E+00
      6      1187.462737756  -2.5934306E+03   2.0184061E+00
      7      1325.907915120   1.3844518E+02   1.8339012E+00
      8      1382.337942615   5.6430027E+01   2.1096857E+00
      9      2031.711269520   6.4937333E+02   2.0500930E+00
     10      642.8574052129  -1.3888539E+03   1.7616232E+00
     11      324.8009486357  -3.1805646E+02   1.6155105E+00
     12      1082.015764327   7.5721482E+02   1.8760200E+00
     13      396.1978599574  -6.8581790E+02   1.6444562E+00
     14      529.6231461584   1.334252

np.True_

## Visualize the modeling results

Here are the POV-Ray settings for rendering the final combined model. The rotation parameter is set to give a top-down perspective that shows both pathways clearly. Render the image and compare it to published mechanistic diagrams of the ORR on Pt(111).

In [ ]:
model = read('model.traj',-1)

In [ ]:
##-Bond detection-##

##-radius multiplies covalent_radii to set the cutoff distance.
##-1.0 = strict covalent bonds; bump to 1.1-1.2 for MOFs with stretched bonds.
bondpairs = get_bondpairs(model, radius=1.1)

generic_projection_settings = {
    'rotation': '-45x, 0y, 0z',
    'radii': 1.1, ##-reduce from 1.1 → atoms don't swallow bonds
    'colors': None,
    'show_unit_cell': 0,
}

povray_settings['textures'] = ['intermediate' for atom in model] ##-try options: jmol, ase2, ase3, vmd, glass, pale, simple
write(f'model.pov', model, **generic_projection_settings, povray_settings=povray_settings)
os.system(f'povray +Imodel.pov +Omodel.png +A +AM2 +UA +Q11 +W4096 +H2048')

0

In [ ]:
img = Image.open('model.png')
ratio = 1.5
img_resized = img.resize((int(img.width * ratio), int(img.height)))
img_mirror = ImageOps.mirror(img_resized)
img_mirror.save('model_fix.png')  ##-Replace with the desired output path

In [ ]:
img_mirror

## Summary

After completing this notebook, you should be able to:

1. Construct oxide surfaces from CIF files with manual atom substitution to create single-atom sites. Build M-N-C active sites by modifying a graphene nanoribbon with del, append, and symbol reassignment. Load molecular catalysts from XYZ files and place adsorbates at geometrically defined positions relative to the active metal center. Construct metal slabs and place multiple ORR intermediates simultaneously.
2. Run optimizations with BFGS for single-site systems and FIRE for larger slabs with multiple adsorbates and loose convergence thresholds.
3. Configure and run POV-Ray rendering. Set camera type, canvas resolution, lighting (area and point lights), depth cueing, and atom textures. Write .pov files from ASE structures using write() with generic_projection_settings and povray_settings, invoke POV-Ray from Python via os.system(), and control output quality and resolution through command-line flags.
4. Render a batch of structures in a loop, resize the output images with Pillow, and assemble them into an animated PNG with the apng library, ready for conversion to GIF.

4. Concatenate independently optimized slabs, adjust unit cell dimensions, add gas-phase molecules at defined positions, and produce a single rendering that captures an entire reaction mechanism.